# Challenge 2: Minería de asteroides — Pipeline 1
## Razonamiento, espacio de estados, exploración, búsqueda y resultados

**Problema:** encontrar una ruta válida para evaluar una estrategia de inversión en minería de asteroides.  
**Tipo de razonamiento:** deductivo: si una condición se cumple, se permite la transición.  
**Datos:** escenarios **sintéticos**, no conclusiones reales de inversión.

Seguimos el orden de la rúbrica: **Problema → Espacio de estados → Exploración → Búsqueda → Resultado**.

### 1. Cargar el espacio de estados
Leemos los archivos JSON que ya están en GitHub. **No necesitas subir archivos manualmente a Colab.**

In [ ]:
import json
from urllib.request import urlopen
from collections import deque
import matplotlib.pyplot as plt

base = "https://raw.githubusercontent.com/pao25-ing/Machine_Learning/main/Challenge_2/data/"
modelo = json.load(urlopen(base + "espacio_estados_condiciones.json"))
escenarios = json.load(urlopen(base + "escenarios_asteroides.json"))["escenarios"]

print("Estados:", len(modelo["estados"]))
print("Transiciones posibles:", len(modelo["transiciones"]))

### 2. Elegir un escenario experimental
Podemos cambiar el nombre por: `evidencia_insuficiente`, `problema_tecnologico`, `economia_desfavorable`, `piloto_desfavorable_revisable` o `datos_incompletos`.

La ruta objetivo de inversión gradual (S16) solo corresponde al escenario favorable. En los demás casos usamos como objetivo su decisión final esperada.

In [ ]:
escenario = "favorable_dos_estrategias"  # Cambiar para probar otro caso

datos = escenarios[escenario]["variables"]
objetivos = {
    "favorable_dos_estrategias": "S16",
    "evidencia_insuficiente": "S15",
    "problema_tecnologico": "S17",
    "economia_desfavorable": "S17",
    "piloto_desfavorable_revisable": "S15",
    "datos_incompletos": "S16"  # No existe ruta: falta la evaluación económica
}
objetivo = objetivos[escenario]
print("Escenario:", escenario)
print("Objetivo:", objetivo)

### 3. Construir el grafo de estados válido
Cada flecha tiene una condición. Si la condición se cumple, agregamos esa ruta al diccionario `grafo`.

- `siempre`: la transición está permitida.
- `igual`: el dato debe coincidir con el valor esperado.
- `en`: el dato debe pertenecer a una lista de opciones.

Si falta un dato, no permitimos avanzar por esa transición.

In [ ]:
grafo = {estado: [] for estado in modelo["estados"]}

for transicion in modelo["transiciones"]:
    regla = transicion["guarda"]
    if regla["tipo"] == "siempre":
        permitido = True
    elif regla["tipo"] == "igual":
        permitido = datos.get(regla["campo"]) == regla["valor"]
    else:  # Tipo "en"
        permitido = datos.get(regla["campo"]) in regla["valores"]

    if permitido:
        grafo[transicion["origen"]].append(transicion["destino"])

for estado, vecinos in grafo.items():
    if vecinos:
        print(estado, "->", vecinos)

### 4. Algoritmo BFS — búsqueda en anchura
Utiliza una **cola (FIFO)**. En este grafo sin pesos, encuentra una ruta con el menor número de transiciones.

In [ ]:
def bfs(grafo, inicio, objetivo):
    cola = deque([[inicio]])
    visitados = []

    while cola:
        ruta = cola.popleft()
        estado = ruta[-1]

        if estado not in visitados:
            visitados.append(estado)
            if estado == objetivo:
                return ruta, visitados
            for vecino in grafo[estado]:
                cola.append(ruta + [vecino])

    return [], visitados

### 5. Algoritmo DFS — búsqueda en profundidad
Utiliza una **pila (LIFO)**. Explora primero una rama del grafo y después regresa a probar otras rutas.

In [ ]:
def dfs(grafo, inicio, objetivo):
    pila = [[inicio]]
    visitados = []

    while pila:
        ruta = pila.pop()
        estado = ruta[-1]

        if estado not in visitados:
            visitados.append(estado)
            if estado == objetivo:
                return ruta, visitados
            for vecino in reversed(grafo[estado]):
                pila.append(ruta + [vecino])

    return [], visitados

### 6. Búsqueda exhaustiva (BF)
En lugar de detenerse en la primera solución, **revisa las rutas completas que terminan en S15, S16 o S17**. No confundir BF (Brute Force) con BFS.

In [ ]:
def busqueda_exhaustiva(grafo, inicio):
    pila = [[inicio]]
    rutas = []
    evaluaciones = 0
    finales = ["S15", "S16", "S17"]

    while pila:
        ruta = pila.pop()
        evaluaciones += 1
        estado = ruta[-1]
        if estado in finales:
            rutas.append(ruta)
        else:
            for vecino in grafo[estado]:
                if vecino not in ruta:
                    pila.append(ruta + [vecino])

    return rutas, evaluaciones

### 7. Comparar resultados
Comparamos la ruta encontrada y la cantidad de estados o rutas examinadas.

In [ ]:
ruta_bfs, vistos_bfs = bfs(grafo, "S0", objetivo)
ruta_dfs, vistos_dfs = dfs(grafo, "S0", objetivo)
rutas_bf, evaluaciones_bf = busqueda_exhaustiva(grafo, "S0")

print("BFS:", " -> ".join(ruta_bfs) if ruta_bfs else "Sin ruta")
print("DFS:", " -> ".join(ruta_dfs) if ruta_dfs else "Sin ruta")
print("BF: rutas completas encontradas:", len(rutas_bf))
for ruta in rutas_bf:
    print("  ", " -> ".join(ruta))

print("Estados visitados por BFS:", len(vistos_bfs))
print("Estados visitados por DFS:", len(vistos_dfs))
print("Evaluaciones de BF:", evaluaciones_bf)

### 8. Gráfico sencillo de comparación
**Nota:** BF cuenta evaluaciones de rutas, por lo que no es exactamente la misma medida que los estados distintos visitados por BFS y DFS.

In [ ]:
nombres = ["BFS", "DFS", "BF"]
valores = [len(vistos_bfs), len(vistos_dfs), evaluaciones_bf]

plt.bar(nombres, valores)
plt.title("Exploración de algoritmos (escenario sintético)")
plt.ylabel("Estados visitados / evaluaciones")
plt.show()

### 9. Interpretación

- **BFS:** muestra una ruta con el menor número de transiciones en este grafo sin pesos.
- **DFS:** depende del orden en que explora las ramas.
- **BF:** enumera todas las rutas terminales habilitadas en el escenario.
- **Límite:** encontrar una ruta corta **no demuestra** que una estrategia sea económicamente más rentable.

El grafo tiene **18 estados y 28 transiciones posibles**, pero en cada escenario solo se exploran las que cumplen sus condiciones. Los datos y las decisiones son **hipotéticos**.